In [19]:
import os
from pathlib import Path
import time
import h5py
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import e, m_p
import pandas as pd

import xtrack as xt
import xpart as xp
import xobjects as xo

from cpymad.madx import Madx

In [20]:
slices = 4
thick = False
threads = 10
folder = Path('Lattice_Files/02_Aperture_Lattice/')

madx = Madx(stdout=False)
madx.call(str(folder / 'ISIS.injected_beam'))
madx.call(str(folder / 'ISIS.elements'))
madx.call(str(folder / 'ISIS.strength'))
madx.call(str(folder / 'ISIS.sequence'))
madx.call(str(folder / 'ISIS.aperture'))
madx.use('synchrotron')
madx.command.select(flag='makethin', slice=slices, thick=thick)
madx.command.makethin(sequence='synchrotron', style='teapot', makedipedge=True)

line = xt.Line.from_madx_sequence(madx.sequence.synchrotron, install_apertures=True)
context = xo.ContextCpu(omp_num_threads=threads)

Converting sequence "synchrotron":   0%|          | 0/1990 [00:00<?, ?it/s]

In [21]:
n_particles = int(2e4)
nemitt_x_0 = 3e-8
nemitt_y_0 = 3e-8

line.set_particle_ref('proton', p0c=0.37033168 * 1e9)
twiss = line.twiss(method='4d', freeze_longitudinal=True,)

particles = line.build_particles(
    num_particles=n_particles,
    x_norm=np.random.normal(size=n_particles),
    y_norm=np.random.normal(size=n_particles),
    px_norm=np.random.normal(size=n_particles),
    py_norm=np.random.normal(size=n_particles),
    nemitt_x=nemitt_x_0,
    nemitt_y=nemitt_y_0,
    method='4d',
    freeze_longitudinal=True,
    mode='normalized_transverse',
)

x_0 = particles.x
y_0 = particles.y
px_0 = particles.px
py_0 = particles.py

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


In [22]:
line.build_tracker(_context=context)
n_turns = 200 # Seem to need arounf 50

start=time.perf_counter()
line.track(particles, num_turns=n_turns, with_progress=True, freeze_longitudinal=True, turn_by_turn_monitor=True)#'ONE_TURN_EBE')
print(time.perf_counter()-start)

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


Tracking:   0%|          | 0/200 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
138.47736642699988


In [23]:
x_tbt = line.record_last_track.x
px_tbt = line.record_last_track.px
y_tbt = line.record_last_track.y
py_tbt = line.record_last_track.py

In [24]:
filename= 'hyper_no_sc_200' #str(input('Please enter a name for this file: '))

df_twiss = twiss.to_pandas()
df_twiss.to_hdf(f'data/{filename}.h5', key='twiss_data', mode='w', format='fixed')

with h5py.File(f'data/{filename}.h5', 'a') as f:
    f.create_dataset('x_tbt', data=x_tbt, compression='gzip')
    f.create_dataset('y_tbt', data=y_tbt, compression='gzip')
    f.create_dataset('px_tbt', data=px_tbt, compression='gzip')
    f.create_dataset('py_tbt', data=py_tbt, compression='gzip')

    f.attrs['time'] = datetime.now().strftime("%Y%m%d_%H%M%S")
    f.attrs['threads'] = threads
    f.attrs['slices'] = slices
    f.attrs['thick'] = thick

    f.attrs['gamma'] = particles.gamma0[0]
    f.attrs['beta'] = particles.beta0[0]
    
    f.attrs['n_particles'] = n_particles
    f.attrs['nemitt_x_0'] = nemitt_x_0
    f.attrs['nemitt_y_0'] = nemitt_y_0

    
    f.attrs['qx'] = twiss.qx
    f.attrs['qy'] = twiss.qy
    f.attrs['dqx'] = twiss.dqx
    f.attrs['dqy'] = twiss.dqy

/tmp/ipykernel_4655/4292285214.py:4: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block3_values] [items->Index(['W_matrix'], dtype='str')]

  df_twiss.to_hdf(f'data/{filename}.h5', key='twiss_data', mode='w', format='fixed')
